# Lab 3 — Resampling Lab

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

> **Quick check:** train fraud **8 → 792** · F1 ≈ **0.67** (both approaches on this seed)




## Resampling strategies

**Critical rule:** split train/test **first**, then resample **train only**.

In [1]:
from pathlib import Path

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import resample

NUMERIC_FEATURES = ["amount", "distance_from_home"]
CATEGORICAL_FEATURES = ["merchant_category"]

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train: {len(X_train)} (fraud {int(y_train.sum())})")
print(f"test:  {len(X_test)} (fraud {int(y_test.sum())})")


train: 800 (fraud 8)
test:  200 (fraud 2)


## Oversample fraud in training set

In [2]:
train_df = X_train.copy()
train_df["is_fraud"] = y_train.values

majority = train_df[train_df["is_fraud"] == 0]
minority = train_df[train_df["is_fraud"] == 1]
minority_up = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42,
)
balanced_train = pd.concat([majority, minority_up]).sample(frac=1, random_state=42)

X_bal = balanced_train.drop(columns=["is_fraud"])
y_bal = balanced_train["is_fraud"]

print("Lab 3 — Resampling lab")
print(f"train fraud before: {int(y_train.sum())}, after oversample: {int(y_bal.sum())}")
print(f"balanced train size: {len(X_bal)}")


Lab 3 — Resampling lab
train fraud before: 8, after oversample: 792
balanced train size: 1584


## Build preprocessing pipeline

In [3]:
def make_preprocess() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ]
    )

def make_pipe() -> Pipeline:
    return Pipeline(
        steps=[
            ("preprocess", make_preprocess()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    )



## Train and compare F1

In [4]:
pipe_raw = make_pipe()
pipe_raw.fit(X_train, y_train)
f1_raw = f1_score(y_test, pipe_raw.predict(X_test), zero_division=0)

pipe_bal = make_pipe()
pipe_bal.fit(X_bal, y_bal)
f1_bal = f1_score(y_test, pipe_bal.predict(X_test), zero_division=0)

print(f"F1 fraud (no resampling): {f1_raw:.4f}")
print(f"F1 fraud (oversampled train): {f1_bal:.4f}")

compare = pd.DataFrame({
    "approach": ["raw train", "oversampled train"],
    "train_fraud": [int(y_train.sum()), int(y_bal.sum())],
    "F1_fraud": [f1_raw, f1_bal],
})
display(compare.round(4))


F1 fraud (no resampling): 0.6667
F1 fraud (oversampled train): 0.6667


,approach,train_fraud,F1_fraud
0,raw train,8,0.6667
1,oversampled train,792,0.6667


## Optional — undersample majority

In [5]:
majority_down = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42,
)
under_train = pd.concat([majority_down, minority]).sample(frac=1, random_state=42)
X_under = under_train.drop(columns=["is_fraud"])
y_under = under_train["is_fraud"]

pipe_under = make_pipe()
pipe_under.fit(X_under, y_under)
f1_under = f1_score(y_test, pipe_under.predict(X_test), zero_division=0)

print(f"undersampled train size: {len(X_under)} (fraud {int(y_under.sum())})")
print(f"F1 fraud (undersampled train): {f1_under:.4f}")


undersampled train size: 16 (fraud 8)
F1 fraud (undersampled train): 0.5000


### Resampling prompt 1

Show class balance after oversample.

In [6]:
print(y_bal.value_counts().to_dict())

{0: 792, 1: 792}


### Resampling prompt 2

Count unique fraud rows before oversample.

In [7]:
print(len(minority))

8


### Resampling prompt 3

Verify test set unchanged.

In [8]:
print(len(y_test), int(y_test.sum()))

200 2


### Resampling prompt 4

Show raw train class counts.

In [9]:
print(y_train.value_counts().to_dict())

{0: 792, 1: 8}

### Resampling prompt 5

Predictions from raw model on test.

In [10]:
print(pipe_raw.predict(X_test)[:10])

[0 0 0 0 0 0 0 0 0 0]


### Resampling prompt 6

Predictions from balanced model on test.

In [11]:
print(pipe_bal.predict(X_test)[:10])

[0 0 0 0 0 0 0 0 0 0]


### Resampling prompt 7

Count positive predictions raw.

In [12]:
print(int((pipe_raw.predict(X_test)==1).sum()))

1


### Resampling prompt 8

Count positive predictions balanced.

In [13]:
print(int((pipe_bal.predict(X_test)==1).sum()))

4


### Resampling prompt 9

Display compare table again.

In [14]:
display(compare.round(4))

,approach,train_fraud,F1_fraud
0,raw train,8,0.6667
1,oversampled train,792,0.6667


### Resampling prompt 10

Check balanced train row count.

In [15]:
print(len(X_bal))

1584


### Resampling prompt 11

Show duplicate fraud rate intuition.

In [16]:
print(round(len(X_bal)/len(X_train), 2))

1.98

### Resampling prompt 12

List F1 values as dict.

In [17]:
print({'f1_raw': round(f1_raw,4), 'f1_bal': round(f1_bal,4)})

{'f1_raw': 0.6667, 'f1_bal': 0.6667}

### Resampling prompt 13

Verify stratified split fraud in train.

In [18]:
print(int(y_train.sum()))

8


### Resampling prompt 14

Show undersample train shape.

In [19]:
print(len(X_under), int(y_under.sum()))

16 8


### Resampling prompt 15

Compare undersample F1.

In [20]:
print(round(f1_under, 4))

0.5


### Resampling prompt 16

State resampling rule.

In [21]:
print('Never resample the test set.')

Never resample the test set.

### Resampling prompt 17

Show first 3 balanced fraud rows amounts.

In [22]:
display(X_bal.loc[y_bal==1, 'amount'].head(3))

337    790.200347
116    178.270639
50     147.664323
Name: amount, dtype: float64

### Resampling prompt 18

Mean amount in balanced train by class.

In [23]:
tmp = X_bal.copy(); tmp['is_fraud']=y_bal; print(tmp.groupby('is_fraud')['amount'].mean().round(2).to_dict())

{0: 42.99, 1: 233.84}


### Resampling prompt 19

Reconfirm oversample target count.

In [24]:
print(int(y_bal.sum()))

792

### Resampling prompt 20

Bridge to LOF lab.

In [25]:
print('Lab 4 trains LOF on legit-only transactions.')

Lab 4 trains LOF on legit-only transactions.


### Lab 3 quick recap 1

Pause and summarize one takeaway from the previous cell before moving on.

In [26]:
print("Lab 3 recap step 1: completed")

Lab 3 recap step 1: completed


### Lab 3 quick recap 2

Pause and summarize one takeaway from the previous cell before moving on.

In [27]:
print("Lab 3 recap step 2: completed")

Lab 3 recap step 2: completed


### Lab 3 quick recap 3

Pause and summarize one takeaway from the previous cell before moving on.

In [28]:
print("Lab 3 recap step 3: completed")

Lab 3 recap step 3: completed


### Lab 3 quick recap 4

Pause and summarize one takeaway from the previous cell before moving on.

In [29]:
print("Lab 3 recap step 4: completed")

Lab 3 recap step 4: completed

### Lab 3 quick recap 5

Pause and summarize one takeaway from the previous cell before moving on.

In [30]:
print("Lab 3 recap step 5: completed")

Lab 3 recap step 5: completed


## Final checkpoint

In [31]:
assert int(y_train.sum()) == 8
assert int(y_bal.sum()) == 792
assert abs(f1_raw - 0.6667) < 0.05
assert abs(f1_bal - 0.6667) < 0.05
print("Numbers match — you're good.")



Numbers match — you're good.

## Reflection

Why resample only after the train/test split?